In [1]:
from ash import *

ashpath: /Users/tom/Documents/ucl/projects/ash-fork/ash
Sys path: ['/Users/tom/Documents/ucl/projects/ash-fork/ash', '/Users/tom/.local/share/uv/python/cpython-3.11.12-macos-aarch64-none/lib/python311.zip', '/Users/tom/.local/share/uv/python/cpython-3.11.12-macos-aarch64-none/lib/python3.11', '/Users/tom/.local/share/uv/python/cpython-3.11.12-macos-aarch64-none/lib/python3.11/lib-dynload', '', '/Users/tom/Documents/ucl/projects/ash-fork/.venv/lib/python3.11/site-packages', '__editable__.ash-0.95.finder.__path_hook__']
--------------------------------------------------------------------------------
--------------------------------------------------------------------------------
                                           ASH                                            
                              A MULTISCALE MODELLING PROGRAM                              
                                        Version: 0.9dev                                        
                               Git c

In [2]:
BASIS = 'sto-3g'
XC = 'b3lyp'
CHARGE = 0
MULT = 1

In [3]:
#Defining fragment
frag = Fragment(xyzfile="system_aftersolvent.xyz", charge=CHARGE, mult=MULT)


--------------------------------------------------------------------------------
                                New ASH fragment                                
--------------------------------------------------------------------------------

ASH Fragment creation
Reading coordinates from XYZ file 'system_aftersolvent.xyz' into fragment.
Creating/Updating fragment attributes...
Number of Atoms in fragment: 2633
Formula: P1O879H1753
Label: system_aftersolvent
Charge: 0 Mult: 1

--------------------------------------------------------------------------------


In [4]:
xyz_list = [i for i in zip(frag.elems, frag.coords)]

lines = [str(len(xyz_list)), '']
for symbol, coords in xyz_list:
    line = f"{symbol} {coords[0]} {coords[1]} {coords[2]}"
    lines.append(line)

xyz_string = "\n".join(lines)

In [8]:
qm_atoms = list(range(8))

In [9]:
qm_atoms

[0, 1, 2, 3, 4, 5, 6, 7]

In [10]:
qm_xyz_list = [
    (frag.elems[i], frag.coords[i])
    for i in qm_atoms
]

lines = [str(len(qm_xyz_list)), '']
for symbol, coords in qm_xyz_list:
    line = f"{symbol} {coords[0]} {coords[1]} {coords[2]}"
    lines.append(line)

qm_xyz_string = "\n".join(lines)
print(qm_xyz_string)

8

P 0.0 0.0 0.0
O -1.472 0.003 0.545
O 0.437 -1.273 -0.984
H -0.316 -1.911 -0.967
O 0.442 1.271 -0.984
H -0.308 1.912 -0.966
O 1.241 -0.002 1.107
H 2.083 -0.004 0.59


In [11]:
N_ACT = 2

In [12]:
nbed_theory = NbedTheory(
    geometry=qm_xyz_string,
    n_active_atoms=N_ACT,
    basis=BASIS,
    xc_functional=XC,
    projector='mu',
    localization='spade',
    # run_ccsd_emb=True
)



                     #####################################                      
                     #                                   #                      
                     #     NbedTheory initialization     #                      
                     #                                   #                      
                     #####################################                      


In [14]:
# water_xml = "/opt/homebrew/Caskroom/miniconda/base/envs/ash-conda/lib/python3.11/site-packages/openmm/app/data/amber14/tip3p.xml"
water_xml = "amber14/tip3p.xml"

frozen_atoms=listdiff(frag.allatoms,qm_atoms)

openmm_theory = OpenMMTheory(
    xmlfiles=["openff_LIG.xml", water_xml], 
    pdbfile="system_aftersolvent.pdb", 
    # periodic=True, 
    # autoconstraints=None,
    # rigidwater=False,
    # frozen_atoms=qm_atoms,
)



                           #########################                            
                           #                       #                            
                           #     OpenMM Theory     #                            
                           #                       #                            
                           #########################                            
OpenMM CPU threads set to: 1
Imported OpenMM library version: 8.3.1

--------------------------------------------------------------------------------
                             Defining OpenMM object                             
--------------------------------------------------------------------------------

Printlevel: 2
HBonds option: X-H bond lengths will automatically be constrained
AutoConstraint setting: HBonds
Rigidwater constraints: True
Hydrogenmass option: 1.5 Da
Using platform: CPU

--------------------------------------------------------------------------------
          

In [15]:
qmmm_theory = QMMMTheory(
    qm_theory = nbed_theory,
    mm_theory = openmm_theory,
    fragment = frag,
    qm_charge = CHARGE,
    qm_mult = MULT,
    qmatoms = qm_atoms,
    printlevel = 3,
)



                            ########################                            
                            #                      #                            
                            #     QM/MM Theory     #                            
                            #                      #                            
                            ########################                            
QM-theory: NbedTheory
MM-theory: OpenMMTheory
All atoms in fragment: 2633
QM region (8 atoms): [0, 1, 2, 3, 4, 5, 6, 7]
MM region (2625 atoms)
QM/MM object selected to use 1 cores
Embedding: elstat
No atomcharges list passed to QMMMTheory object
Getting system charges from OpenMM object
QM-region coordinates (before linkatoms):
   0    P   0.00000000    0.00000000    0.00000000
   1    O  -1.47200000    0.00300000    0.54500000
   2    O   0.43700000   -1.27300000   -0.98400000
   3    H  -0.31600000   -1.91100000   -0.96700000
   4    O   0.44200000    1.27100000   -0.98400000
   5    

In [16]:
qmmm_theory

In [18]:
MolecularDynamics(
    fragment=frag,
    theory=qmmm_theory,
    timestep=0.001,
    simulation_steps=20,
    traj_frequency=1,
    temperature=300,
    # integrator='LangevinIntegrator',
    coupling_frequency=1,
    charge=CHARGE,
    mult=MULT
)



                     ######################################                     
                     #                                    #                     
                     #     OpenMM MD wrapper function     #                     
                     #                                    #                     
                     ######################################                     


              ####################################################              
              #                                                  #              
              #     OpenMM Molecular Dynamics Initialization     #              
              #                                                  #              
              ####################################################              
Analyzing theory input to OpenMM_MDclass
This is an QMMMTheory object
Turning on externalforce option.
Added force
System is non-periodic. Setting enforcePeriodicBox to False

----------

In [ ]:
# Optimizer(
#     fragment=frag, 
#     theory=qmmm_theory, 
#     ActiveRegion=True, 
#     actatoms=qm_atoms
# )